In [ ]:
!pip install pyreadr
!pip install polars

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.3/418.3 kB 14.7 MB/s eta 0:00:00


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not in Colab environment. Assuming local file access.")
    # If not in Colab, BASE_DIR should point to a local path
    # For this script, we'll assume the /content/drive path works


Mounted at /content/drive
Google Drive mounted successfully.


In [ ]:
# =================== TEMP EXCLUDE SECTION (remove later) ===================
from pathlib import Path
import csv
import os

BASE_DIR = '/content/drive/MyDrive/Data_LM/CRSP_Compustat_data'


PARQUET_DIR_STR = os.path.join(BASE_DIR, 'parquet_data') # Keep string version for os.path.join

# 1) Hand-pick big files to skip entirely (by full path or by filename)
TEMP_EXCLUDE_PATHS = {
}
TEMP_EXCLUDE_NAMES = {
    # or exclude by name anywhere:
    # 'StkDlySecurityData.rds', 'sfz_dp_dly.rds', 'sfz_ds_dly.rds'
}

# 2) Also skip any source already converted (using the manifest)
USE_MANIFEST_EXCLUDE = True
# Use Path for MANIFEST_PATH definition
MANIFEST_PATH = Path(PARQUET_DIR_STR) / '_manifest.csv'


already_converted_sources = set()
if USE_MANIFEST_EXCLUDE and MANIFEST_PATH.exists():
    with open(MANIFEST_PATH, newline='') as f:
        for row in csv.DictReader(f):
            if row.get('status') in ('ok', 'skipped_exists'):
                # row['source'] is an absolute path we recorded earlier
                try:
                    already_converted_sources.add(Path(row['source']))
                except Exception:
                    pass

# 3) Or: skip sources if a newer/equal Parquet exists (no manifest needed)
EXCLUDE_IF_PARQUET_UPTODATE = True

def _has_up_to_date_parquet(rpath: Path) -> bool:
    rel_parent = rpath.parent.relative_to(BASE_DIR)
    out_dir = Path(PARQUET_DIR_STR) / rel_parent # Use Path(PARQUET_DIR_STR) here
    base = rpath.stem
    # handle both single-object and multi-object naming
    candidates = list(out_dir.glob(f"{base}.parquet")) + list(out_dir.glob(f"{base}__*.parquet"))
    if not candidates:
        return False
    src_mtime = rpath.stat().st_mtime
    return all(p.stat().st_mtime >= src_mtime for p in candidates)

def should_skip(rpath: Path) -> bool:
    if rpath in TEMP_EXCLUDE_PATHS:                           # explicit path
        return True
    if rpath.name in TEMP_EXCLUDE_NAMES:                      # explicit name
        return True
    if rpath in already_converted_sources:                    # from manifest
        return True
    if EXCLUDE_IF_PARQUET_UPTODATE and _has_up_to_date_parquet(rpath):
        return True
    return False
# ================= END TEMP EXCLUDE SECTION (remove later) =================


# === Bulk convert .rds -> Polars -> Parquet (recursive, mirrored dirs) ===
import sys, os, gc, csv
from pathlib import Path
import pandas as pd
import polars as pl
import pyreadr

# --- Colab mount (safe no-op outside Colab) ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# === CONFIG ===
BASE_DIR = Path('/content/drive/MyDrive/Data_LM/CRSP_Compustat_data')

# All source directories you showed in the screenshot:
SOURCE_DIRS = [
    #BASE_DIR / 'documents-export-2025-3-19',
    BASE_DIR / 'documents-export-2025-3-19(1)',
    BASE_DIR / 'documents-export-2025-3-19(2)',
]

PARQUET_DIR = BASE_DIR / 'parquet_data'            # output root
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

SKIP_EXISTING = True                                # don't re-write if up-to-date
ROW_GROUP_SIZE = 100_000                            # parquet row group size (tune as needed)
COMPRESSION = 'zstd'                                # 'zstd' is a great default
HANDLE_TZ_AS_UTC = True                             # strip tz to naive UTC for portability
CAST_CATEGORIES_TO_STRING = True                    # pandas categorical -> string

# === HELPERS ===
def discover_r_files(dirs):
    exts = ('.rds', '.Rds', '.RDS', '.rda', '.RData')
    for d in dirs:
        if not Path(d).exists():
            continue
        for p in Path(d).rglob('*'):
            if p.suffix in exts and p.is_file():
                yield p

def out_dir_for(src_path: Path) -> Path:
    # Mirror the source directory structure under PARQUET_DIR
    rel_parent = src_path.parent.relative_to(BASE_DIR)
    return PARQUET_DIR / rel_parent

def sanitize_pandas(df_pd: pd.DataFrame) -> pd.DataFrame:
    # Optional: normalize dtypes for smoother pandas->polars conversion
    if CAST_CATEGORIES_TO_STRING:
        for c in df_pd.select_dtypes(['category']).columns:
            df_pd[c] = df_pd[c].astype('string')

    if HANDLE_TZ_AS_UTC:
        for c in df_pd.columns:
            if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
                # convert to UTC and drop tz (parquet+cross-lang friendly)
                df_pd[c] = df_pd[c].dt.tz_convert('UTC').dt.tz_localize(None)
    return df_pd

def write_parquet(df_pl: pl.DataFrame, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    df_pl.write_parquet(
        str(out_path),
        compression=COMPRESSION,
        statistics=True,
        row_group_size=ROW_GROUP_SIZE
    )

# === MAIN ===
manifest = []
# files = list(discover_r_files(SOURCE_DIRS))
files = [p for p in discover_r_files(SOURCE_DIRS) if not should_skip(Path(p))] # Cast p to Path
print(f"Planning to process {len(files)} files after exclusions.")

print(f"Discovered {len(files)} R files.")

for rpath in files:
    df_pd = None # Initialize df_pd to None at the start of each file loop
    try:
        res = pyreadr.read_r(str(rpath))  # may return multiple objects
    except Exception as e:
        manifest.append({
            'source': str(rpath), 'object': None, 'rows': None, 'cols': None,
            'target': None, 'status': 'read_error', 'msg': str(e)
        })
        continue

    if len(res) == 0:
        manifest.append({
            'source': str(rpath), 'object': None, 'rows': None, 'cols': None,
            'target': None, 'status': 'empty_r_file', 'msg': 'no objects inside'
        })
        continue

    for obj_name, df_pd in res.items():
        try:
            # pandas -> tidy up -> polars
            df_pd = sanitize_pandas(df_pd)
            out_subdir = out_dir_for(Path(rpath)) # Cast rpath to Path
            base_name = Path(rpath).stem  # file name without extension, cast rpath to Path
            # If an .rds contains multiple objects, distinguish by suffix --> None in file names cause of this
            # out_name = f"{base_name}__{obj_name}.parquet"
            keys = list(res.keys())
            if len(keys) == 1 and (keys[0] is None or str(keys[0]).lower() == 'none'):
                out_name = f"{base_name}.parquet"
            else:
                out_name = f"{base_name}__{obj_name or 'obj'} .parquet"

            out_path = out_subdir / out_name

            # Skip if an up-to-date Parquet already exists
            if SKIP_EXISTING and out_path.exists():
                if out_path.stat().st_mtime >= Path(rpath).stat().st_mtime: # Cast rpath to Path
                    manifest.append({
                        'source': str(rpath), 'object': obj_name,
                        'rows': getattr(df_pd, 'shape', (None, None))[0],
                        'cols': getattr(df_pd, 'shape', (None, None))[1],
                        'target': str(out_path), 'status': 'skipped_exists',
                        'msg': ''
                    })
                    # delete df_pd only after it's used for the manifest entry
                    del df_pd
                    continue

            df_pl = pl.from_pandas(df_pd, include_index=False, nan_to_null=True)
            write_parquet(df_pl, out_path)

            manifest.append({
                'source': str(rpath), 'object': obj_name,
                'rows': df_pl.height, 'cols': df_pl.width,
                'target': str(out_path), 'status': 'ok', 'msg': ''
            })
            # delete df_pd only after it's used for the manifest entry
            del df_pd
        except Exception as e:
            manifest.append({
                'source': str(rpath), 'object': obj_name,
                'rows': getattr(df_pd, 'shape', (None, None))[0] if df_pd is not None else None,
                'cols': getattr(df_pd, 'shape', (None, None))[1] if df_pd is not None else None,
                'target': None, 'status': 'write_error', 'msg': str(e)
            })
            # delete df_pd after it's used for the manifest entry in case of error
            if df_pd is not None:
                del df_pd
        finally:
            # free df_pl memory aggressively between files
            try: del df_pl
            except: pass
            gc.collect()

# Write a small manifest so you can audit what happened
man_path = Path(PARQUET_DIR_STR) / '_manifest.csv' # Use Path(PARQUET_DIR_STR) here
fieldnames = sorted({k for row in manifest for k in row.keys()})
with open(man_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(manifest)

n_ok = sum(1 for m in manifest if m['status'] == 'ok')
n_skip = sum(1 for m in manifest if m['status'] == 'skipped_exists')
n_err = sum(1 for m in manifest if 'error' in m['status'])
print(f"Done. ok={n_ok}, skipped={n_skip}, errors={n_err}. Manifest: {man_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Planning to process 29 files after exclusions.
Discovered 29 R files.


/tmp/ipython-input-3799251662.py:132: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
/tmp/ipython-input-3799251662.py:132: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
/tmp/ipython-input-3799251662.py:132: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
/tmp/ipython-input-3799251662.py:132: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
/tmp/ipython

Done. ok=29, skipped=0, errors=0. Manifest: /content/drive/MyDrive/Data_LM/CRSP_Compustat_data/parquet_data/_manifest.csv


/tmp/ipython-input-3799251662.py:132: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if pd.api.types.is_datetime64tz_dtype(df_pd[c].dtype):
